# Inspecting the residual `T1ce - T1`The quantity `residual_mode` actually supervises. Four questions:1. **What does it look like?** (section 1)2. **How is it distributed?** (section 2) -- the histogram, whole-brain and split by   enhancing tumour.3. **Where does its energy sit, and what does that imply for `et_weight`?** (section 3)   `training.losses.weighted_loss` counts ET pixels `et_weight` times as heavily; the right   value depends on how concentrated the residual already is, which is measurable rather   than assumable. The configs currently ship `et_weight = 330`, derived from a *homogeneous   error* approximation -- section 3 replaces that assumption with the real number.4. **How much does it vary slice to slice?** (section 4) -- the variance that made the old   area-normalized ET term so unstable.Section 5 optionally loads a trained checkpoint and overlays the *predicted* residual on theground-truth one: the visual form of the `resid_ratio` collapse detector.> Related but different: `inspect_enhancement.ipynb` asks whether enhancement is additive or> multiplicative, using ET voxels only. This notebook treats the residual as a **supervision> target across the whole brain**.

## 0. Setup

In [ ]:
import os, sys, jsonimport numpy as npimport torchimport matplotlib.pyplot as plt%matplotlib inline# ---- EDIT: repo root -------------------------------------------------------------------REPO = "/scratch/ee2178/ImMAP"if os.path.isdir(REPO):    os.chdir(REPO)elif os.path.basename(os.getcwd()) == "notebooks":       # running in place    os.chdir(os.path.dirname(os.getcwd()))sys.path.insert(0, os.getcwd())# ---- EDIT: which config supplies the data block ----------------------------------------# Any synthesis config works, as does a trained run's saved trained_nets/.../config.json.CFG_PATH = "config/BraTS/residual_synthesis_dtcdlnet.json"SPLIT    = "val"CROP     = 192      # deterministic CENTER crop, so every rerun sees the same pixelsBATCH    = 8N_BATCH  = 40       # batches pooled for the statistics (40 x 8 = 320 slices)SEED     = 0with open(CFG_PATH) as f:    cfg = json.load(f)SRC_IDX = int(cfg.get("training", {}).get("residual_src_idx", 1))   # the T1 anchorprint("cwd:", os.getcwd())print(f"config      : {CFG_PATH}")print(f"input_idx   : {cfg['data'][SPLIT]['input_idx']}   (stored order: flair, t1, t1ce, t2)")print(f"target_idx  : {cfg['data'][SPLIT]['target_idx']}")print(f"anchor      : input channel {SRC_IDX}  ->  residual = target - X[:, {SRC_IDX}]")

In [ ]:
import datasets                                  # registers the loadersfrom datasets.registry import build_loaderbase = dict(cfg["data"][SPLIT])base.update(name="synthesis", scales=None, et_mask=True,            center_crop=CROP, random_flips=False, num_workers=0, batch_size=BATCH)base.pop("crop_size", None)                      # random crop -> the center crop set aboveloader = build_loader(base, shuffle=True, drop_last=False)print(f"{len(loader.dataset):,} slices in the {SPLIT} split")def batches(n):    """First `n` batches of a fresh pass. Seed the global RNG first for a reproducible    sample -- shuffle=True draws its permutation from it."""    it = iter(loader)    for _ in range(n):        try:            yield next(it)        except StopIteration:            returndef residual_of(X, y):    return y - X[:, SRC_IDX:SRC_IDX + 1]torch.manual_seed(SEED)X, y, mask, et = next(iter(loader))resid = residual_of(X, y)print(f"X {tuple(X.shape)}  y {tuple(y.shape)}  mask {tuple(mask.shape)}  et {tuple(et.shape)}")print(f"residual range over brain: [{float((resid*mask).min()):+.2f}, "      f"{float((resid*mask).max()):+.2f}]  (z-scored units)")

## 1. What it looks likeThe residual is near zero over most of the brain -- T1 already carries the bulk anatomy --and departs from zero at strong edges and in the enhancing tumour. A diverging colormap(white = 0) is the only honest way to render a mostly-zero signed map; grayscale wouldshow it as flat mid-tone and hide the sign.

In [ ]:
k = int(et.sum(dim=(1, 2, 3)).argmax())          # the slice with the most enhancing tumourt1   = (X[k, SRC_IDX] * mask[k, 0]).numpy()t1ce = (y[k, 0]       * mask[k, 0]).numpy()r    = (resid[k, 0]   * mask[k, 0]).numpy()etk  = et[k, 0].numpy()vmax = float(np.abs(r).max())fig, ax = plt.subplots(1, 4, figsize=(17, 4.4))ax[0].imshow(t1, cmap="gray");   ax[0].set_title("T1 (anchor)")ax[1].imshow(t1ce, cmap="gray"); ax[1].set_title("T1ce (target)")im = ax[2].imshow(r, cmap="bwr", vmin=-vmax, vmax=vmax)ax[2].set_title(f"residual  T1ce - T1   (+/- {vmax:.2f})")fig.colorbar(im, ax=ax[2], fraction=0.046)ax[3].imshow(t1ce, cmap="gray")ax[3].imshow(np.ma.masked_where(etk == 0, etk), cmap="autumn", alpha=0.85)ax[3].set_title(f"ET overlay  ({int(etk.sum())} px)")for a in ax:    a.axis("off")plt.tight_layout(); plt.show()

## 2. The histogramPooled over `N_BATCH` batches. Two views:* **left** -- every brain voxel, log counts. Expect a tall spike at 0 with heavy tails.* **right** -- inside vs outside the ET as *densities*. Densities, not counts: ET is a few  hundred voxels against a few hundred thousand, so raw counts would render the ET  histogram as a flat line at the bottom of the axis.

In [ ]:
torch.manual_seed(SEED)r_in, r_out = [], []for Xb, yb, mb, eb in batches(N_BATCH):    rb    = residual_of(Xb, yb)    brain = mb.bool().expand_as(rb)    etb   = (eb * mb).bool().expand_as(rb)    r_in.append(rb[etb].numpy())    r_out.append(rb[brain & ~etb].numpy())r_in  = np.concatenate(r_in)  if r_in  else np.zeros(0, np.float32)r_out = np.concatenate(r_out) if r_out else np.zeros(0, np.float32)r_all = np.concatenate([r_out, r_in])print(f"pooled: {r_out.size:,} non-ET brain voxels, {r_in.size:,} ET voxels")

In [ ]:
lim  = float(np.percentile(np.abs(r_all), 99.9))bins = np.linspace(-lim, lim, 201)fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))ax[0].hist(r_all, bins=bins, color="0.35")ax[0].set_yscale("log"); ax[0].axvline(0, color="k", lw=0.8)ax[0].set_title(f"residual over all brain voxels  (n = {r_all.size:,})")ax[0].set_xlabel("T1ce - T1   (z-scored units)"); ax[0].set_ylabel("count (log)")ax[1].hist(r_out, bins=bins, density=True, alpha=0.6, color="0.45",           label=f"non-ET brain  (n = {r_out.size:,})")if r_in.size:    ax[1].hist(r_in, bins=bins, density=True, alpha=0.6, color="crimson",               label=f"ET  (n = {r_in.size:,})")ax[1].set_yscale("log"); ax[1].axvline(0, color="k", lw=0.8)ax[1].set_title("density, inside vs outside the enhancing tumour")ax[1].set_xlabel("T1ce - T1"); ax[1].set_ylabel("density (log)"); ax[1].legend()plt.tight_layout(); plt.show()def describe(v, name):    if not v.size:        print(f"{name:<15s} (empty)"); return    q = np.percentile(v, [1, 25, 50, 75, 99])    print(f"{name:<15s} n={v.size:>10,}  mean={v.mean():+.3f}  sd={v.std():.3f}  "          f"rms={np.sqrt((v**2).mean()):.3f}  |  p1={q[0]:+.2f}  p25={q[1]:+.2f}  "          f"p50={q[2]:+.2f}  p75={q[3]:+.2f}  p99={q[4]:+.2f}")describe(r_all, "all brain")describe(r_out, "non-ET brain")describe(r_in,  "ET")

## 3. Where the energy lives, and what `et_weight` should be`weighted_loss` computes `mean_i(w_i * err_i)` with `w = 1 + (et_weight - 1) * et`, dividingby the **total** pixel count. So the tumour's share of the objective is$$\text{share}(w) \;=\; \frac{w\,E_{\mathrm{ET}}}{E_{\mathrm{out}} + w\,E_{\mathrm{ET}}},\qquadw(\text{share}) \;=\; \frac{s\,E_{\mathrm{out}}}{(1-s)\,E_{\mathrm{ET}}}$$with `E` the summed squared error in each region. Two caveats worth stating plainly:* This measures the residual **target**, i.e. the error of a predictor that outputs zero --  where training *starts*. As the model fits the easy bulk first, the ET's share of the  *remaining* error rises on its own, so this is a floor, not a fixed point.* `et_weight = 330` in the configs came from assuming the error is spread evenly  (`share = f*w / (1-f+f*w)` at `f = 0.003`). If the residual concentrates in the tumour --  and it should -- the honest number is **much lower**. A value below 1 would mean the  residual already concentrates there more than you are asking the loss to.

In [ ]:
def residual_energy(r, mask, et):    """Energy bookkeeping for a signed residual r (B, 1, H, W).    N is the FULL pixel count, matching what the loss divides by (`region.numel()` in    weighted_loss); background contributes nothing because r is brain-masked. Both area    fractions are reported: f_img drives the loss arithmetic, f_brain is the    anatomically meaningful one."""    r   = r * mask    etb = (et * mask).bool().expand_as(r)    A   = float(etb.sum())    E_et  = float((r[etb] ** 2).sum()) if A else 0.0    E_tot = float((r ** 2).sum())    return dict(N=float(r.numel()), N_brain=float(mask.expand_as(r).sum()), A=A,                E_et=E_et, E_out=E_tot - E_et, E_tot=E_tot)def et_share(w, E_et, E_out):    return w * E_et / max(E_out + w * E_et, 1e-12)def weight_for_share(s, E_et, E_out):    return s * E_out / max((1.0 - s) * E_et, 1e-12)torch.manual_seed(SEED)tot = dict(N=0.0, N_brain=0.0, A=0.0, E_et=0.0, E_out=0.0, E_tot=0.0)per_slice = []for Xb, yb, mb, eb in batches(N_BATCH):    rb = residual_of(Xb, yb)    s  = residual_energy(rb, mb, eb)    for key in tot:        tot[key] += s[key]    for i in range(rb.shape[0]):                      # kept for section 4        per_slice.append(residual_energy(rb[i:i+1], mb[i:i+1], eb[i:i+1]))f_img   = tot["A"] / tot["N"]f_brain = tot["A"] / max(tot["N_brain"], 1.0)share   = tot["E_et"] / max(tot["E_tot"], 1e-12)print(f"ET area        : {f_img*100:.3f}% of the image,  {f_brain*100:.3f}% of the brain")print(f"ET residual    : {share*100:.2f}% of the total squared residual energy")print(f"concentration  : {share / max(f_brain, 1e-12):.1f}x   "      f"(1x = residual spread evenly over the brain)")

In [ ]:
print(f"{'et_weight':>10}  {'ET share of the loss':>21}")for w in (1, 2, 5, 10, 30, 50, 100, 330):    print(f"{w:>10}  {et_share(w, tot['E_et'], tot['E_out'])*100:20.1f}%")print(f"\n{'target share':>13}  {'et_weight':>10}")for s_t in (0.10, 0.20, 0.30, 0.50):    print(f"{s_t*100:12.0f}%  {weight_for_share(s_t, tot['E_et'], tot['E_out']):10.1f}")ws = np.logspace(0, 3, 200)plt.figure(figsize=(6.5, 4))plt.semilogx(ws, [et_share(w, tot["E_et"], tot["E_out"]) * 100 for w in ws], lw=2)plt.axvline(330, color="crimson", ls="--", lw=1,            label="330 (config default, homogeneous-error estimate)")for s_t in (10, 30):    plt.axhline(s_t, color="0.7", ls=":", lw=1)plt.xlabel("et_weight  (per-ET-pixel multiplier)")plt.ylabel("ET share of the objective  (%)")plt.title("what et_weight buys, measured on this split")plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Slice-to-slice variabilityThis is the distribution the old area-normalized ET term divided by. `region_loss` scaledthe per-ET-pixel gradient as `1/|ET|`, so the spread below translated directly into a spreadin per-pixel weight -- and many slices carry no enhancing tumour at all, on which the oldterm returned 0 and the step descended a different objective entirely. The per-pixel weightform has none of that: the multiplier is `et_weight` on every ET pixel of every slice.

In [ ]:
A_i     = np.array([s["A"] for s in per_slice])share_i = np.array([s["E_et"] / max(s["E_tot"], 1e-12) for s in per_slice])n_zero  = int((A_i == 0).sum())print(f"{len(A_i)} slices:  {n_zero} ({n_zero/len(A_i)*100:.0f}%) contain NO enhancing tumour")nz = A_i > 0if nz.any():    q = np.percentile(A_i[nz], [5, 25, 50, 75, 95])    print(f"ET area on slices that have any (px):  p5={q[0]:.0f}  p25={q[1]:.0f}  "          f"p50={q[2]:.0f}  p75={q[3]:.0f}  p95={q[4]:.0f}")    print(f"  -> under the OLD 1/|ET| normalization those p5 and p95 slices differed by "          f"{q[4]/max(q[0], 1):.0f}x in per-pixel gradient weight, on top of any real "          f"difference in error.")fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))ax[0].hist(A_i, bins=40, color="0.4"); ax[0].set_yscale("log")ax[0].set_title("ET area per slice"); ax[0].set_xlabel("ET pixels")ax[0].set_ylabel("slices (log)")if nz.any():    ax[1].scatter(A_i[nz], share_i[nz] * 100, s=14, alpha=0.5, color="crimson")    ax[1].set_xscale("log")ax[1].set_xlabel("ET pixels in slice")ax[1].set_ylabel("ET share of that slice's residual energy (%)")ax[1].set_title("bigger tumour -> more of the residual")plt.tight_layout(); plt.show()

## 5. Optional: the model's predicted residualSet `RUN_CFG` to a trained run's saved `config.json` (its `paths.ckpt` must exist). Overlaysthe predicted residual's histogram on the ground truth's and reports `resid_ratio =rms(pred)/rms(gt)`, globally and restricted to the ET.`resid_ratio -> 0` is the collapse this task is prone to: predicting zero reconstructs T1ceas exactly the T1 anchor, which PSNR and SSIM both reward because T1 already carries thebulk anatomy. On the histogram it shows up as a predicted distribution far narrower thanthe target's.

In [ ]:
# ---- EDIT: a trained run's saved config, or None to skip --------------------------------RUN_CFG = None    # e.g. "trained_nets/brats/Residual_DT_CDLNet_dS_T1ce_ET/config.json"if RUN_CFG is None:    print("RUN_CFG is None -- skipping. Point it at a trained run's config.json to compare.")else:    from training.common import load_model    from training.synthesis import predict    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    net = load_model(RUN_CFG, device=device)    torch.manual_seed(SEED)    g_all, p_all, g_et, p_et = [], [], [], []    with torch.no_grad():        for Xb, yb, mb, eb in batches(N_BATCH):            Xb, yb, mb, eb = (t.to(device) for t in (Xb, yb, mb, eb))            gb    = residual_of(Xb, yb) * mb            pb    = predict(net, Xb) * mb            brain = mb.bool().expand_as(gb)            etb   = (eb * mb).bool().expand_as(gb)            g_all.append(gb[brain].cpu().numpy()); p_all.append(pb[brain].cpu().numpy())            g_et.append(gb[etb].cpu().numpy());    p_et.append(pb[etb].cpu().numpy())    g_all = np.concatenate(g_all); p_all = np.concatenate(p_all)    g_et  = np.concatenate(g_et);  p_et  = np.concatenate(p_et)    rms = lambda v: float(np.sqrt((v ** 2).mean())) if v.size else float("nan")    print(f"resid_ratio   (whole brain): {rms(p_all)/max(rms(g_all), 1e-8):.3f}")    print(f"resid_ratio   (ET only)    : {rms(p_et)/max(rms(g_et), 1e-8):.3f}")    print("  1.0 = right magnitude,  -> 0 = the net has stopped predicting enhancement")    lim  = float(np.percentile(np.abs(g_all), 99.9))    bins = np.linspace(-lim, lim, 201)    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))    for a, (g, p, ttl) in zip(ax, [(g_all, p_all, "whole brain"), (g_et, p_et, "ET only")]):        a.hist(g, bins=bins, density=True, alpha=0.55, color="0.35", label="ground truth")        a.hist(p, bins=bins, density=True, alpha=0.55, color="tab:blue", label="predicted")        a.set_yscale("log"); a.axvline(0, color="k", lw=0.8)        a.set_title(f"residual density -- {ttl}")        a.set_xlabel("T1ce - T1"); a.set_ylabel("density (log)"); a.legend()    plt.tight_layout(); plt.show()